# nbplay — Complete Demo

This notebook demonstrates every widget and the **Session** workflow:

1. **Settings** — audio device &amp; MIDI configuration
2. **Synths** — oscillator widgets with waveform display
3. **Sampler** — sample loading with ADSR envelope
4. **Step Sequencers** — per-track step patterns
5. **Transport** — global play/stop, BPM, time signature, loop
6. **Session + Tracks** — connect everything with auto-routed mixer
7. **Mixer** — gain (dB), pan, mute/solo per channel
8. **Bridge to Rust** — convert widget state to Rust objects

In [8]:
import ipywidgets as widgets
from nbplay import (
    SynthWidget,
    SettingsWidget,
    SequencerWidget,
    SamplerWidget,
    MixerWidget,
    TransportWidget,
    Track,
    Session,
)

## 1. Audio & MIDI Settings

The `SettingsWidget` lets users select their audio output device
and connect MIDI controllers via the Web MIDI API.

In [9]:
settings = SettingsWidget()
settings

## 2. Create Synths

Two oscillator widgets: a **saw** topline and a **square** bassline.
Double-click the frequency or amplitude labels to edit inline.

In [10]:
topline = SynthWidget(
    oscillator_type="saw",
    frequency=880.0,
    amplitude=0.6,
)

bassline = SynthWidget(
    oscillator_type="square",
    frequency=110.0,
    amplitude=0.7,
)

widgets.HBox([topline, bassline], layout=widgets.Layout(gap="16px"))

## 3. Create a Sampler

Load a short synthetic click and shape it with an ADSR envelope.
The attack is clamped to a 5 ms minimum so the gain ramp always completes.

In [11]:
import math

sampler = SamplerWidget(
    attack=0.001,
    decay=0.05,
    sustain=0.0,
    release=0.05,
    max_voices=4,
)

# Generate a short sine click (100 ms at 44.1 kHz)
sr = 44100
duration = 0.1
n_samples = int(sr * duration)
click_data = [
    math.sin(2 * math.pi * 1000 * i / sr) * max(0, 1 - i / n_samples)
    for i in range(n_samples)
]
sampler.load_sample(click_data, sample_rate=sr, root_note=60, name="Click")

sampler

## 4. Step Sequencers

Each sound source gets a sequencer. The topline plays an ascending arpeggio;
the bassline pulses on the root with accented downbeats.

In [12]:
topline_seq = SequencerWidget(length=8, bpm=120.0)
for i, note in enumerate([72, 76, 79, 81, 79, 76, 72, 69]):
    topline_seq.set_step(i, note=note, velocity=100, active=True)

bassline_seq = SequencerWidget(length=8, bpm=120.0)
for i, (note, vel) in enumerate([
    (48, 120), (48, 60), (48, 80), (48, 60),
    (48, 110), (48, 60), (48, 80), (48, 60),
]):
    bassline_seq.set_step(i, note=note, velocity=vel, active=True)

# Sampler sequencer — trigger on beats 1 and 3
sampler_seq = SequencerWidget(length=8, bpm=120.0)
for i in [0, 4]:
    sampler_seq.set_step(i, note=60, velocity=110, active=True)

widgets.HBox(
    [topline_seq, bassline_seq, sampler_seq],
    layout=widgets.Layout(gap="16px"),
)

## 5. Session Workflow (recommended)

A `Session` connects a shared `TransportWidget` and `MixerWidget` to tracks.
BPM and play state automatically propagate from transport → sequencers.
Audio routes through the mixer via `session_id` and `channel_index`.

In [13]:
session = Session(bpm=120.0, time_signature=(4, 4))

# Add tracks — each gets a sequencer, sound source, and auto-created mixer channel
session.add_track("Topline", topline_seq, topline)
session.add_track("Bassline", bassline_seq, bassline)
session.add_track("Sampler", sampler_seq, sampler)

# Adjust mixer — gain is displayed in dB, double-click to edit
session.mixer.set_channel_gain(0, 0.7)   # ~−3.1 dB
session.mixer.set_channel_pan(0, 0.3)    # slightly right
session.mixer.set_channel_gain(1, 0.9)   # ~−0.9 dB
session.mixer.set_channel_pan(1, -0.2)   # slightly left
session.mixer.set_channel_gain(2, 0.5)   # −6.0 dB
session.mixer.master_gain = 0.85

print(session)

Session(bpm=120.0, tracks=3, channels=3)


## 6. Dashboard

Display the transport, sequencers, synths, sampler, and mixer together.
Press ▶ on the transport to start playback — all sequencers sync automatically.

In [15]:
first_col = widgets.VBox(
    [session.transport, topline, bassline],
    layout=widgets.Layout(gap="16px"),
)

second_col = widgets.VBox(
    [topline_seq, bassline_seq, sampler_seq],
    layout=widgets.Layout(gap="16px"),
)

third_col = widgets.VBox(
    [sampler, session.mixer],
    layout=widgets.Layout(gap="16px"),
)

bottom_row = widgets.HBox(
    [settings],
    layout=widgets.Layout(gap="16px"),
)

dashboard = widgets.VBox(
    [
        widgets.HTML(
            "<h2 style='color:#00d4ff;font-family:monospace'>🎹 nbplay Studio</h2>"
        ),
        widgets.HBox(
            [first_col, second_col, third_col],
            layout=widgets.Layout(gap="16px"),
        ),
        bottom_row
    ],
    layout=widgets.Layout(gap="16px", padding="16px"),
)

dashboard

## 7. Bridge to Rust

Convert widget state to Rust-backed objects for offline audio rendering.

In [11]:
rust_mixer = session.mixer.to_mixer()
topline_pattern = topline_seq.to_pattern()
bassline_pattern = bassline_seq.to_pattern()
sampler_pattern = sampler_seq.to_pattern()
rust_sampler = sampler.to_sampler()

print(f"Mixer channels:         {len(rust_mixer)}")
print(f"Topline pattern length: {len(topline_pattern)}")
print(f"Bassline pattern length:{len(bassline_pattern)}")
print(f"Sampler pattern length: {len(sampler_pattern)}")
print(f"Sampler active voices:  {rust_sampler.active_voice_count()}")

Mixer channels:         3
Topline pattern length: 8
Bassline pattern length:8
Sampler pattern length: 8
Sampler active voices:  0
